## Function 7: Overlay & Visualize 🗺️

In this notebook, you'll learn how to build the `overlay_and_visualize()` function step by step. This is where you combine spatial layers using overlay operations and create visual outputs to communicate your results clearly.

### 🎯 What This Function Does
- Perform intersection, union, and difference overlays
- Create matplotlib visualizations
- Calculate overlay statistics
- Save maps to files

### 🔧 Function Signature
```python
def overlay_and_visualize(gdf1, gdf2=None, overlay_how='intersection', save_path=None):
    """
    Args:
        gdf1 (geopandas.GeoDataFrame): First spatial dataset
        gdf2 (geopandas.GeoDataFrame, optional): Second spatial dataset
        overlay_how (str): Overlay operation to perform
        save_path (str or Path, optional): File path for saving the figure
    
    Returns:
        dict: Overlay result, figure, and statistics
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/spatial_basics.py`  
**Function Name**: `overlay_and_visualize()`  
**Replace**: The placeholder function with your working code

---

### 📚 Step 1: Load Data for Overlay Operations

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# Load datasets
ecoregions = gpd.read_file('../data/ecoregions/epa_level3_western_us.geojson')
parks = gpd.read_file('../data/protected_areas/national_parks_major.geojson')

# Get a subset for clearer visualization
parks_subset = parks.iloc[:5]  # First 5 parks
eco_subset = ecoregions.iloc[:3]  # First 3 ecoregions

print(f"✅ Loaded:")
print(f"   Ecoregions: {len(ecoregions)} total ({len(eco_subset)} for examples)")
print(f"   Parks: {len(parks)} total ({len(parks_subset)} for examples)")

---

### 🗺️ Step 2: Visualization 1 - Overlay Operations Grid

**Level 4 Goal:** Master ALL overlay types - see how each creates different results


In [ ]:
# Perform all 4 overlay operations
try:
    intersection = gpd.overlay(parks_subset, eco_subset, how='intersection')
    union = gpd.overlay(parks_subset, eco_subset, how='union')
    difference = gpd.overlay(parks_subset, eco_subset, how='difference')
    symmetric_difference = gpd.overlay(parks_subset, eco_subset, how='symmetric_difference')
    
    # Create 2x3 grid
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.ravel()
    
    # 1. ORIGINAL DATA
    eco_subset.plot(ax=axes[0], color='lightblue', edgecolor='blue', linewidth=2, alpha=0.6, label='Ecoregions')
    parks_subset.plot(ax=axes[0], color='lightgreen', edgecolor='darkgreen', linewidth=2, alpha=0.6, label='Parks')
    axes[0].legend(fontsize=10, framealpha=0.9)
    axes[0].set_title("Original Data\nInput Layers", fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].grid(True, alpha=0.3)
    
    # 2. INTERSECTION (overlap only)
    eco_subset.plot(ax=axes[1], color='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
    parks_subset.plot(ax=axes[1], color='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
    if len(intersection) > 0:
        intersection.plot(ax=axes[1], color='yellow', edgecolor='red', linewidth=2, alpha=0.8)
    axes[1].set_title(f"Intersection\n{len(intersection)} features (overlap only)", fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Longitude')
    axes[1].set_ylabel('Latitude')
    axes[1].grid(True, alpha=0.3)
    axes[1].text(0.5, 0.02, "Where parks AND ecoregions overlap",
                transform=axes[1].transAxes, ha='center', va='bottom',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8), fontsize=9)
    
    # 3. UNION (everything)
    if len(union) > 0:
        union.plot(ax=axes[2], color='purple', edgecolor='black', linewidth=1, alpha=0.5, cmap='Set3')
    axes[2].set_title(f"Union\n{len(union)} features (everything)", fontsize=13, fontweight='bold')
    axes[2].set_xlabel('Longitude')
    axes[2].set_ylabel('Latitude')
    axes[2].grid(True, alpha=0.3)
    axes[2].text(0.5, 0.02, "All parks OR ecoregions (combined)",
                transform=axes[2].transAxes, ha='center', va='bottom',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=9)
    
    # 4. DIFFERENCE (A minus B)
    eco_subset.plot(ax=axes[3], color='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
    parks_subset.plot(ax=axes[3], color='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
    if len(difference) > 0:
        difference.plot(ax=axes[3], color='orange', edgecolor='darkorange', linewidth=2, alpha=0.7)
    axes[3].set_title(f"Difference\n{len(difference)} features (parks - ecoregions)", fontsize=13, fontweight='bold')
    axes[3].set_xlabel('Longitude')
    axes[3].set_ylabel('Latitude')
    axes[3].grid(True, alpha=0.3)
    axes[3].text(0.5, 0.02, "Parks that DON'T overlap ecoregions",
                transform=axes[3].transAxes, ha='center', va='bottom',
                bbox=dict(boxstyle='round', facecolor='orange', alpha=0.8), fontsize=9)
    
    # 5. SYMMETRIC DIFFERENCE (A xor B)
    eco_subset.plot(ax=axes[4], color='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
    parks_subset.plot(ax=axes[4], color='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
    if len(symmetric_difference) > 0:
        symmetric_difference.plot(ax=axes[4], color='pink', edgecolor='purple', linewidth=2, alpha=0.7)
    axes[4].set_title(f"Symmetric Difference\n{len(symmetric_difference)} features (non-overlapping)", fontsize=13, fontweight='bold')
    axes[4].set_xlabel('Longitude')
    axes[4].set_ylabel('Latitude')
    axes[4].grid(True, alpha=0.3)
    axes[4].text(0.5, 0.02, "Everything EXCEPT overlap",
                transform=axes[4].transAxes, ha='center', va='bottom',
                bbox=dict(boxstyle='round', facecolor='pink', alpha=0.8), fontsize=9)
    
    # 6. SUMMARY TABLE
    axes[5].axis('off')
    summary_text = "📊 OVERLAY OPERATIONS SUMMARY\n\n"
    summary_text += f"Original Parks: {len(parks_subset)} features\n"
    summary_text += f"Original Ecoregions: {len(eco_subset)} features\n\n"
    summary_text += "Results:\n"
    summary_text += f"• Intersection: {len(intersection)} (overlap only)\n"
    summary_text += f"• Union: {len(union)} (everything)\n"
    summary_text += f"• Difference: {len(difference)} (parks not in ecoregions)\n"
    summary_text += f"• Sym Diff: {len(symmetric_difference)} (non-overlapping parts)\n\n"
    summary_text += "Use cases:\n"
    summary_text += "• Intersection → Find shared areas\n"
    summary_text += "• Union → Merge datasets\n"
    summary_text += "• Difference → Remove overlaps\n"
    summary_text += "• Sym Diff → Find exclusive areas"
    
    axes[5].text(0.5, 0.5, summary_text,
                transform=axes[5].transAxes,
                ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black', linewidth=2),
                fontsize=11, family='monospace')
    
    plt.suptitle("Complete Overlay Operations Guide", fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
    
    print(f"🎯 Overlay operations completed:")
    print(f"   Intersection: {len(intersection)} features")
    print(f"   Union: {len(union)} features")
    print(f"   Difference: {len(difference)} features")
    print(f"   Symmetric Difference: {len(symmetric_difference)} features")
    
except Exception as e:
    print(f"⚠️  Overlay demonstration skipped: {str(e)}")
    print("   (This can happen if geometries don't overlap enough)")


---

### 🗺️ Step 3: Visualization 2 - Professional Styled Map

**Level 4 Goal:** Create publication-quality cartography with legends, titles, and annotations

In [ ]:
# Load full datasets for professional map
cities = gpd.read_file('../data/cities/ne_cities_us.geojson')

# Create professional-quality figure
fig, ax = plt.subplots(figsize=(16, 12))

# Base layer - ecoregions with subtle colors
ecoregions.plot(ax=ax, 
               column='US_L3CODE' if 'US_L3CODE' in ecoregions.columns else None,
               cmap='Pastel1',
               edgecolor='black',
               linewidth=0.8,
               alpha=0.6,
               legend=False)

# Parks layer - prominent green
parks.plot(ax=ax,
          color='#2d7f2e',  # Forest green
          edgecolor='#1a4d1a',
          linewidth=1.5,
          alpha=0.7,
          label='National Parks')

# Cities layer - prominent red
cities.plot(ax=ax,
           color='#e74c3c',  # Professional red
           markersize=25,
           alpha=0.8,
           edgecolor='white',
           linewidth=0.5,
           label='Cities',
           zorder=5)

# Professional title with subtitle
title = "Western US Ecoregions with National Parks and Cities"
subtitle = "Multi-layer Spatial Analysis | GeoPandas Visualization"
ax.text(0.5, 1.05, title,
       transform=ax.transAxes,
       ha='center', va='bottom',
       fontsize=16, fontweight='bold')
ax.text(0.5, 1.02, subtitle,
       transform=ax.transAxes,
       ha='center', va='bottom',
       fontsize=11, style='italic', color='gray')

# Axis labels
ax.set_xlabel('Longitude (degrees)', fontsize=12, fontweight='bold')
ax.set_ylabel('Latitude (degrees)', fontsize=12, fontweight='bold')

# Zoom to ecoregions extent (Western US focus)
minx, miny, maxx, maxy = ecoregions.total_bounds
# Add small buffer for aesthetics (5% padding)
x_buffer = (maxx - minx) * 0.05
y_buffer = (maxy - miny) * 0.05
ax.set_xlim(minx - x_buffer, maxx + x_buffer)
ax.set_ylim(miny - y_buffer, maxy + y_buffer)

# Professional gridlines
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5, color='gray')
ax.set_axisbelow(True)  # Grid behind data

# Custom legend with counts
legend_elements = [
    Line2D([0], [0], marker='o', color='w', 
          markerfacecolor='#2d7f2e', markersize=12, 
          label=f'National Parks ({len(parks)})', linewidth=2, markeredgecolor='#1a4d1a'),
    Line2D([0], [0], marker='o', color='w', 
          markerfacecolor='#e74c3c', markersize=8, 
          label=f'Cities ({len(cities)})', markeredgecolor='white'),
    Rectangle((0, 0), 1, 1, facecolor='lightblue', edgecolor='black', alpha=0.6,
             label=f'Ecoregions ({len(ecoregions)})')
]
legend = ax.legend(handles=legend_elements,
                  loc='lower left',
                  fontsize=11,
                  framealpha=0.95,
                  edgecolor='black',
                  title='Legend',
                  title_fontsize=12)
legend.get_frame().set_linewidth(1.5)

# Add north arrow (simple text-based)
ax.text(0.95, 0.95, '▲\nN',
       transform=ax.transAxes,
       ha='center', va='top',
       fontsize=14, fontweight='bold',
       bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='black', linewidth=1.5))

# Add data source citation
citation = "Data: Natural Earth, EPA Ecoregions, National Park Service | Created with GeoPandas & Matplotlib"
ax.text(0.5, -0.05, citation,
       transform=ax.transAxes,
       ha='center', va='top',
       fontsize=8, style='italic', color='gray')

# Add statistics box
stats_text = f"📊 Dataset Statistics\n\n"
stats_text += f"Ecoregions: {len(ecoregions)}\n"
stats_text += f"Parks: {len(parks)}\n"
stats_text += f"Cities: {len(cities)}\n\n"
stats_text += f"Total Area: {ecoregions.geometry.area.sum()/1e10:.1f} × 10¹⁰ m²"

ax.text(0.98, 0.02, stats_text,
       transform=ax.transAxes,
       ha='right', va='bottom',
       bbox=dict(boxstyle='round,pad=0.7', facecolor='white', alpha=0.95, edgecolor='black', linewidth=1.5),
       fontsize=9, family='monospace')

plt.tight_layout()
plt.show()

print(f"✅ Professional map created!")
print(f"   Quality: Publication-ready")
print(f"   Features: Styled layers, legend, north arrow, citation, statistics")
print(f"   This demonstrates professional cartographic standards")


---

### 🗺️ Step 4: Visualization 3 - Interactive Web Map

**Level 4 Goal:** Create an interactive, web-ready map with zoom, pan, and popups

Using **Folium** (Leaflet.js for Python) to create an interactive web map:

In [ ]:
import folium
from folium import GeoJson, Marker, Circle

# Calculate center of western US
center_lat = (ecoregions.total_bounds[1] + ecoregions.total_bounds[3]) / 2
center_lon = (ecoregions.total_bounds[0] + ecoregions.total_bounds[2]) / 2

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles='OpenStreetMap',
    control_scale=True
)

# Add ecoregions layer
folium.GeoJson(
    ecoregions,
    name='Ecoregions',
    style_function=lambda x: {
        'fillColor': 'lightblue',
        'color': 'blue',
        'weight': 1,
        'fillOpacity': 0.4
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['eco_name', 'area_km2'] if 'eco_name' in ecoregions.columns else ['area_km2'],
        aliases=['Ecoregion:', 'Area (km²):'],
        localize=True
    )
).add_to(m)

# Add parks layer
folium.GeoJson(
    parks,
    name='National Parks',
    style_function=lambda x: {
        'fillColor': '#2d7f2e',
        'color': '#1a4d1a',
        'weight': 2,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'state', 'area_km2'] if 'name' in parks.columns else ['area_km2'],
        aliases=['Park:', 'State:', 'Area (km²):'],
        localize=True
    )
).add_to(m)

# Add cities as markers
for idx, city in cities.iterrows():
    folium.CircleMarker(
        location=[city.geometry.y, city.geometry.x],
        radius=5,
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7,
        popup=folium.Popup(
            f"<b>{city['name']}</b><br>" +
            (f"State: {city['state_province']}<br>" if 'state_province' in city else "") +
            (f"Population: {city['population']:,}" if 'population' in city else ""),
            max_width=200
        ),
        tooltip=city['name'] if 'name' in city else f'City {idx}'
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add title
title_html = '''
    <div style="position: fixed; 
                top: 10px; left: 50px; width: 400px; height: 60px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:16px; padding: 10px">
    <b>Western US: Ecoregions, Parks & Cities</b><br>
    <i style="font-size:12px;">🟦 Ecoregions | 🟩 Parks | 🔴 Cities</i>
    </div>
'''
m.get_root().html.add_child(folium.Element(title_html))

print("🗺️ Interactive map created!")
print(f"   Features: {len(ecoregions)} ecoregions, {len(parks)} parks, {len(cities)} cities")
print(f"   Interactions: Pan, zoom, click for details, toggle layers")
print(f"\n💡 Try these interactions:")
print(f"   • Zoom in/out with mouse wheel or +/- buttons")
print(f"   • Pan by dragging the map")
print(f"   • Hover over features to see names")
print(f"   • Click cities to see detailed popups")
print(f"   • Use layer control (top right) to toggle layers on/off")
print(f"\n📱 This map is web-ready and mobile-friendly!")

# Display the map
m


In [ ]:
# Multi-panel figure showing complete spatial analysis workflow
fig = plt.figure(figsize=(20, 14))

# Create grid layout
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Panel 1: Base Data (top left)
ax1 = fig.add_subplot(gs[0, 0])
ecoregions.plot(ax=ax1, color='lightblue', edgecolor='black', linewidth=0.5, alpha=0.5)
ax1.set_title("1. Ecoregions\nBase geographic data", fontsize=11, fontweight='bold')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.2)

# Panel 2: Points of Interest (top middle)
ax2 = fig.add_subplot(gs[0, 1])
cities.plot(ax=ax2, color='red', markersize=20, alpha=0.6)
ax2.set_title(f"2. Cities\n{len(cities)} point features", fontsize=11, fontweight='bold')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.2)

# Panel 3: Protected Areas (top right)
ax3 = fig.add_subplot(gs[0, 2])
parks.plot(ax=ax3, color='green', edgecolor='darkgreen', linewidth=1, alpha=0.6)
ax3.set_title(f"3. National Parks\n{len(parks)} polygon features", fontsize=11, fontweight='bold')
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.2)

# Panel 4: Spatial Join (middle left)
ax4 = fig.add_subplot(gs[1, 0])
ecoregions.plot(ax=ax4, color='lightgray', edgecolor='black', linewidth=0.5, alpha=0.3)
cities_joined = gpd.sjoin(cities, ecoregions, how='left', predicate='within')
if 'US_L3NAME' in cities_joined.columns:
    matched = cities_joined[cities_joined['US_L3NAME'].notna()]
    unmatched = cities_joined[cities_joined['US_L3NAME'].isna()]
    matched.plot(ax=ax4, color='green', markersize=30, alpha=0.7)
    if len(unmatched) > 0:
        unmatched.plot(ax=ax4, color='red', markersize=30, alpha=0.7, marker='x')
ax4.set_title(f"4. Spatial Join\nCities + Ecoregion attributes", fontsize=11, fontweight='bold')
ax4.set_xlabel('Longitude')
ax4.set_ylabel('Latitude')
ax4.grid(True, alpha=0.2)

# Panel 5: Overlay Analysis (middle middle)
ax5 = fig.add_subplot(gs[1, 1])
try:
    overlay_result = gpd.overlay(parks_subset, eco_subset, how='intersection')
    eco_subset.plot(ax=ax5, color='lightblue', edgecolor='blue', linewidth=1, alpha=0.3)
    parks_subset.plot(ax=ax5, color='lightgreen', edgecolor='green', linewidth=1, alpha=0.3)
    if len(overlay_result) > 0:
        overlay_result.plot(ax=ax5, color='yellow', edgecolor='red', linewidth=2, alpha=0.8)
    ax5.set_title(f"5. Overlay (Intersection)\n{len(overlay_result)} overlapping areas", fontsize=11, fontweight='bold')
except:
    ax5.text(0.5, 0.5, "Overlay\nAnalysis",
            transform=ax5.transAxes,
            ha='center', va='center',
            fontsize=14, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax5.set_title("5. Overlay Analysis", fontsize=11, fontweight='bold')
ax5.set_xlabel('Longitude')
ax5.set_ylabel('Latitude')
ax5.grid(True, alpha=0.2)

# Panel 6: Buffer Analysis (middle right)
ax6 = fig.add_subplot(gs[1, 2])
cities_sample = cities.iloc[:10]
cities_sample.plot(ax=ax6, color='red', markersize=40, zorder=3)
cities_sample_utm = cities_sample.to_crs('EPSG:32611')
buffer_50km = cities_sample_utm.copy()
buffer_50km.geometry = cities_sample_utm.geometry.buffer(50000)
buffer_50km = buffer_50km.to_crs(cities.crs)
buffer_50km.plot(ax=ax6, alpha=0.2, color='blue', edgecolor='blue', linewidth=1)
ax6.set_title("6. Buffer Analysis\n50km radius zones", fontsize=11, fontweight='bold')
ax6.set_xlabel('Longitude')
ax6.set_ylabel('Latitude')
ax6.grid(True, alpha=0.2)

# Panel 7: Integrated Analysis (bottom - spans all columns)
ax7 = fig.add_subplot(gs[2, :])
ecoregions.plot(ax=ax7, color='lightgray', edgecolor='black', linewidth=0.5, alpha=0.4)
parks.plot(ax=ax7, color='green', edgecolor='darkgreen', linewidth=1, alpha=0.6, label='Parks')
cities.plot(ax=ax7, color='red', markersize=15, alpha=0.7, label='Cities', edgecolor='white', linewidth=0.3)

ax7.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax7.set_title("7. INTEGRATED SPATIAL ANALYSIS\nAll Layers Combined - Complete Geographic Context", 
             fontsize=13, fontweight='bold', pad=15)
ax7.set_xlabel('Longitude (degrees)', fontsize=11)
ax7.set_ylabel('Latitude (degrees)', fontsize=11)
ax7.grid(True, alpha=0.3)

# Add workflow annotation
workflow_text = "Spatial Analysis Workflow:\n"
workflow_text += "1→2→3: Load data layers\n"
workflow_text += "4→5→6: Apply operations\n"
workflow_text += "7: Integrated result"
ax7.text(0.98, 0.98, workflow_text,
        transform=ax7.transAxes,
        ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9),
        fontsize=9, family='monospace')

# Overall title
fig.suptitle("Complete GeoPandas Spatial Analysis Pipeline", 
            fontsize=18, fontweight='bold', y=0.995)

plt.show()

print(f"✅ Multi-panel analysis figure complete!")
print(f"   Panels: 7 (workflow from data loading to integration)")
print(f"   Story: Load → Analyze → Combine → Visualize")
print(f"   This demonstrates how to present spatial analysis results professionally")


scr

### 🏗️ Step 5: Building the Complete Function

Now let's put everything together into a reusable function. This is what you will implement in `src/spatial_basics.py`.

## 🎯 Function Signature

```python
def overlay_and_visualize(gdf1: gpd.GeoDataFrame, gdf2: Optional[gpd.GeoDataFrame] = None, overlay_how: str = 'intersection', save_path: Optional[Union[str, Path]] = None) -> Dict[str, Any]:
```

**Returns:** Dictionary with 'overlay_result' (GeoDataFrame), 'figure' (Figure), 'statistics' (dict)

## 💡 Implementation Tips

1. Start with the function signature from `src/spatial_basics.py`
2. Read the docstring carefully - it explains everything
3. Look at the test file to see what's expected
4. Use the examples above as reference
5. Test frequently as you implement

**Need help?** Check the README or ask on the course forum!

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from typing import Dict, Any, Optional, Union
from pathlib import Path

def overlay_and_visualize(
    gdf1: gpd.GeoDataFrame,
    gdf2: Optional[gpd.GeoDataFrame] = None,
    overlay_how: str = 'intersection',
    save_path: Optional[Union[str, Path]] = None
) -> Dict[str, Any]:
    """
    Perform overlay operations and create visualizations.
    
    Combines two spatial datasets using geometric overlay operations
    and generates both static and interactive visualizations.
    
    Overlay operations:
    - 'intersection': Areas where both datasets overlap
    - 'union': Combined areas from both datasets
    - 'difference': Areas in gdf1 not in gdf2
    - 'symmetric_difference': Areas in either but not both
    
    Args:
        gdf1: First GeoDataFrame
        gdf2: Second GeoDataFrame (optional for visualization only)
        overlay_how: Type of overlay operation
        save_path: Optional path to save the visualization
        
    Returns:
        Dictionary containing:
        - 'overlay_result': GeoDataFrame with overlay results (if gdf2 provided)
        - 'figure': Matplotlib figure object
        - 'statistics': Dictionary of geometry counts and areas
        
    Raises:
        ValueError: If inputs are invalid or overlay operation fails
        
    Example:
        >>> # Find intersection of two polygon layers
        >>> result = overlay_and_visualize(parks, flood_zones, 'intersection')
        >>> overlay_gdf = result['overlay_result']
        >>> print(f"Intersection area: {overlay_gdf.geometry.area.sum()}")
        
        >>> # Just visualize a single layer
        >>> result = overlay_and_visualize(cities, save_path='cities_map.png')
        >>> plt.show()
    """
    import matplotlib.pyplot as plt
    
    result = {}
    
    # If gdf2 is provided, perform overlay operation
    if gdf2 is not None:
        # Validate inputs
        if not isinstance(gdf1, gpd.GeoDataFrame) or not isinstance(gdf2, gpd.GeoDataFrame):
            raise ValueError("Both inputs must be GeoDataFrames")
        
        # Check CRS compatibility
        if gdf1.crs != gdf2.crs:
            raise ValueError(
                f"CRS mismatch: gdf1 has {gdf1.crs}, gdf2 has {gdf2.crs}. "
                "Transform to same CRS before overlay."
            )
        
        # Validate overlay operation
        valid_operations = ['intersection', 'union', 'difference', 'symmetric_difference']
        if overlay_how not in valid_operations:
            raise ValueError(
                f"Invalid overlay operation '{overlay_how}'. "
                f"Must be one of: {', '.join(valid_operations)}"
            )
        
        # Perform overlay
        overlay_result = gpd.overlay(gdf1, gdf2, how=overlay_how)
        result['overlay_result'] = overlay_result
        
        # Calculate statistics
        stats = {
            'operation': overlay_how,
            'input1_count': len(gdf1),
            'input2_count': len(gdf2),
            'output_count': len(overlay_result),
        }
        
        # Calculate areas if geometry is polygons and result is not empty
        if len(overlay_result) > 0:
            geom_type = overlay_result.geometry.geom_type.iloc[0]
            if geom_type in ['Polygon', 'MultiPolygon']:
                # Use projected CRS for area calculation if in geographic CRS
                if overlay_result.crs and overlay_result.crs.is_geographic:
                    overlay_proj = overlay_result.to_crs('EPSG:6933')  # Equal Area
                    total_area_km2 = overlay_proj.geometry.area.sum() / 1e6
                else:
                    total_area_km2 = overlay_result.geometry.area.sum() / 1e6
                stats['total_area_km2'] = float(total_area_km2)
        
        result['statistics'] = stats
        
        # Create visualization
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        if len(overlay_result) > 0:
            overlay_result.plot(ax=ax, alpha=0.7, edgecolor='black', cmap='viridis')
        else:
            # Plot original datasets if overlay is empty
            gdf1.plot(ax=ax, alpha=0.5, edgecolor='red', label='GDF1')
            gdf2.plot(ax=ax, alpha=0.5, edgecolor='blue', label='GDF2')
            ax.legend()
        ax.set_title(f"Overlay Result: {overlay_how.title()}", fontsize=14)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        plt.tight_layout()
        
    else:
        # Visualization only (no overlay)
        result['statistics'] = {
            'feature_count': len(gdf1),
            'geometry_types': gdf1.geometry.geom_type.unique().tolist()
        }
        
        # Create visualization
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        gdf1.plot(ax=ax, alpha=0.7, edgecolor='black', cmap='viridis')
        ax.set_title("Spatial Data Visualization", fontsize=14)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        plt.tight_layout()
    
    result['figure'] = fig
    
    # Save figure if path provided
    if save_path:
        save_path = Path(save_path)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        result['saved_path'] = str(save_path)
    
    return result


# Helper Functions (provided for you)

### ✨ Step 6: Test Your Function

Use these checks before moving on:

- Run the function with two GeoDataFrames and confirm an overlay result is returned
- Try different overlay operations such as **`intersection`**, **`union`**, and **`difference`**
- Confirm the returned dictionary includes **`overlay_result`**, **`figure`**, and **`statistics`**
- Run the function with only one GeoDataFrame and confirm it creates a visualization without overlay
- Test mismatched CRS and confirm it raises a **ValueError**
- Test an invalid overlay type and confirm it raises a **ValueError**
- Test `save_path` and confirm the figure is written to disk

---

### 🧪 Step 7: Verify with Pytest

After implementing in `../src/spatial_basics.py`:

```bash
uv run pytest tests/ -k "TestOverlayAndVisualize" -v
```

**Check your score:**
```bash
uv run python scripts/calculate_grade.py .
```

**⚠️ IMPORTANT: Make sure this passes before you move on!**

---

### 🔑 Key Learning Points

- **`gpd.overlay()`** performs boolean overlay operations between spatial layers
- **`intersection`** keeps only overlapping areas
- **`union`** combines all areas from both datasets
- **`difference`** keeps areas in one dataset that do not overlap the other
- **`symmetric_difference`** keeps non-overlapping parts from both datasets
- Overlay results can be summarized with counts and area statistics
- Good visualization helps communicate spatial analysis clearly and professionally